In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR    = os.path.join(PROJECT_ROOT, 'models')

# Load data
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train_with_cate.parquet'))

with open(os.path.join(PROCESSED_DIR, 'col_definitions.json')) as f:
    cols = json.load(f)

with open(os.path.join(PROCESSED_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)

with open(os.path.join(PROCESSED_DIR, 'dcn_results.json')) as f:
    dcn_results = json.load(f)

Y_col  = cols['Y_col']
T_col  = cols['T_col']
W_cols = cols['W_cols']

feature_cols = [T_col] + W_cols
X = train[feature_cols].fillna(0).values
y = train[Y_col].values

print(f'Loaded: {train.shape}')
print(f'Optuna version: {optuna.__version__}')
print(f'Features: {len(feature_cols)}')

c:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: (500000, 21)
Optuna version: 3.6.1
Features: 10


In [3]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import time

# ─── Train/test split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Use a subsample for HPO — faster trials, same signal
# Economics: this is like running a pilot study before full data collection
X_hpo = X_train[:100_000]
y_hpo = y_train[:100_000]

print(f'HPO sample: {X_hpo.shape} (subsample for faster trials)')
print(f'Final eval: {X_test.shape} (held-out test set)')

# ─── Optuna objective function ────────────────────────────────────────────────
def objective(trial):
    """
    Optuna minimises this function.
    Each call = one trial = one set of hyperparameters evaluated.
    
    Economics analogy: like a grid search over the hyperparameter space,
    but Optuna uses Bayesian optimisation (Tree-structured Parzen Estimator)
    to focus trials on promising regions — much more efficient than grid search.
    """
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves':    trial.suggest_int('num_leaves', 15, 127),
        'max_depth':     trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':     trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':     trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':    trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_jobs': -1,
        'verbose': -1,
        'random_state': 42,
    }

    model = LGBMRegressor(**params)

    # 3-fold CV on HPO subsample — MAPE as objective
    scores = cross_val_score(
        model, X_hpo, y_hpo,
        cv=3,
        scoring='neg_mean_absolute_percentage_error',
        n_jobs=-1,
    )
    return -scores.mean()  # Optuna minimises, so negate


# ─── Run optimisation ─────────────────────────────────────────────────────────
print('\nRunning Optuna — 50 trials of LightGBM HPO...')
print('(Each trial = 3-fold CV on 100k rows — ~5-8 minutes total)')

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)

start = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
elapsed = time.time() - start

print(f'\n✅ HPO complete in {elapsed/60:.1f} minutes')
print(f'\nBest trial:')
print(f'  MAPE (CV):     {study.best_value:.4f}')
print(f'  Parameters:')
for k, v in study.best_params.items():
    print(f'    {k:<25} {v}')

HPO sample: (100000, 10) (subsample for faster trials)
Final eval: (100000, 10) (held-out test set)

Running Optuna — 50 trials of LightGBM HPO...
(Each trial = 3-fold CV on 100k rows — ~5-8 minutes total)


Best trial: 46. Best value: 0.419876: 100%|██████████| 50/50 [20:55<00:00, 25.12s/it]


✅ HPO complete in 20.9 minutes

Best trial:
  MAPE (CV):     0.4199
  Parameters:
    n_estimators              267
    learning_rate             0.03171795214313538
    num_leaves                90
    max_depth                 9
    min_child_samples         51
    subsample                 0.6085091722650441
    colsample_bytree          0.6041944796288204
    reg_alpha                 0.7466351454547987
    reg_lambda                0.010131263028401205


In [ ]:
# ─── Train best model on full training set ────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({'n_jobs': -1, 'verbose': -1, 'random_state': 42})

best_model = LGBMRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
best_mape = mean_absolute_percentage_error(y_test, y_pred)
best_r2   = r2_score(y_test, y_pred)

# ─── Feature importance ───────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_,
}).sort_values('importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance plot
axes[0].barh(importance_df['feature'], importance_df['importance'],
             color='#2563EB', alpha=0.8)
axes[0].set_xlabel('Feature Importance (gain)')
axes[0].set_title('LightGBM Feature Importance\n(Tuned model)')

# Optuna optimisation history
trial_values = [t.value for t in study.trials]
running_best = [min(trial_values[:i+1]) for i in range(len(trial_values))]
axes[1].plot(range(1, len(trial_values)+1), trial_values,
             alpha=0.3, color='#2563EB', label='Trial MAPE')
axes[1].plot(range(1, len(running_best)+1), running_best,
             color='#EA580C', lw=2, label='Best so far')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('MAPE')
axes[1].set_title('Optuna Optimisation History\n(TPE Bayesian search)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'hpo_results.png'), dpi=150, bbox_inches='tight')
plt.show()

# ─── Full model comparison ────────────────────────────────────────────────────
print('=' * 60)
print('FULL MODEL COMPARISON')
print('=' * 60)
print(f'{"Model":<30} {"MAPE":>8} {"R²":>8}')
print('-' * 60)
print(f'{"Ridge (baseline)":<30} {baseline["ridge_mape"]:>8.4f} {baseline["ridge_r2"]:>8.4f}')
print(f'{"DCN (deep learning)":<30} {dcn_results["dcn_mape"]:>8.4f} {dcn_results["dcn_r2"]:>8.4f}')
print(f'{"LightGBM (tuned)":<30} {best_mape:>8.4f} {best_r2:>8.4f}  ← best')
print('-' * 60)
print(f'\nLightGBM improvement over Ridge:')
print(f'  MAPE: {(baseline["ridge_mape"] - best_mape) / baseline["ridge_mape"]:+.1%}')
print(f'  R²:   {best_r2 - baseline["ridge_r2"]:+.4f}')

print(f'\nTop 3 most important features:')
for _, row in importance_df.tail(3).iterrows():
    print(f'  {row["feature"]:<25} {row["importance"]:,.0f}')

# ─── Save everything ──────────────────────────────────────────────────────────
import joblib
joblib.dump(best_model, os.path.join(MODELS_DIR, 'lgbm_tuned.pkl'))

hpo_results = {
    'best_mape': round(best_mape, 6),
    'best_r2': round(best_r2, 6),
    'best_params': best_params,
    'n_trials': len(study.trials),
    'hpo_time_minutes': round(elapsed/60, 1),
}
with open(os.path.join(PROCESSED_DIR, 'hpo_results.json'), 'w') as f:
    json.dump(hpo_results, f, indent=2)

print(f'\n✅ Tuned LightGBM saved  → models/lgbm_tuned.pkl')
print(f'✅ HPO results saved     → data/processed/hpo_results.json')
print(f'\nNotebook 06 complete — ready for notebook 08 (model evaluation)')

FULL MODEL COMPARISON
Model                              MAPE       R²
------------------------------------------------------------
Ridge (baseline)                 0.4292   0.0176
DCN (deep learning)              0.4243   0.0319
LightGBM (tuned)                 0.4177   0.0553  ← best
------------------------------------------------------------

LightGBM improvement over Ridge:
  MAPE: +2.7%
  R²:   +0.0377

Top 3 most important features:
  price_vs_category         5,058
  log_price                 5,230
  log_popularity            5,669

✅ Tuned LightGBM saved  → models/lgbm_tuned.pkl
✅ HPO results saved     → data/processed/hpo_results.json

Notebook 06 complete — ready for notebook 08 (model evaluation)


: 